<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB17_Choosing_an_Architecture_Closing_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB17 · Class 17 — Choosing an Architecture: Closing Block 3**

## Block 3: AI — Deep Learning (closing)

`NB11`–`NB16` each introduced one architecture on one real problem. The question this closing class actually answers isn't "what is a CNN" — it's **"given a new, unseen problem, which of these tools do you reach for, and why?"** We build an explicit decision framework, apply it to every real dataset used so far in this course, and then **prove** one of its central claims with a real experiment: training a neural network on a dataset small enough that classical Machine Learning (`NB10`) should still win — `checking whether it actually does, rather than just asserting it`.

### Learning objectives

By the end of this class, students will be able to:
- Summarize what each Block 3 architecture is for, in one sentence each.
- Apply a concrete decision framework (data type, label availability, dataset size) to a new problem.
- Explain, with real evidence, why Deep Learning does not automatically beat classical ML — and when it does.
- Compare a neural network against a tuned classical model fairly, on the same real data and split.
- Describe, at a high level, what's coming in Block 4.

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap, today's roadmap | 5 min | Theory |
| 2 | Synthesis: everything from Block 3 in one table | 15 min | Theory |
| 3 | A decision framework for choosing an architecture | 15 min | Theory + Practice |
| 4 | Applying the framework to every dataset used this block | 15 min | Theory + Practice |
| 5 | Hands-on: does a neural network beat NB10's tuned Gradient Boosting? | 20 min | Practice |
| 6 | Interpreting the result | 15 min | Practice |
| 7 | When Deep Learning *does* win: revisiting the evidence | 10 min | Theory |
| 8 | Looking ahead: Block 4 | 10 min | Theory |
| 9 | A checklist for approaching a new problem | 10 min | Theory |
| 10 | Summary, homework | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.


---

## 1. Recap: where we are

Six classes, six real problems, one closing question: knowing all of this, how do you actually *decide* what to build next time, before you've spent two hours building the wrong thing?

---

## 2. Synthesis: everything from Block 3 in one table

| Class | Architecture | Core idea | Real data used |
|---|---|---|---|
| `NB11` | Perceptron / MLP | Weighted sums + non-linear activations, stacked in layers | Sonar (mine vs. rock) |
| `NB12` | Training discipline | Validation monitoring, dropout, early stopping, optimizer choice | Sonar |
| `NB13` | CNN | Shared, learned filters that respect spatial structure | LIACi underwater images |
| `NB14` | RNN / LSTM | A cell reused across time steps, carrying a hidden state forward | Ship fuel monthly sequences |
| `NB15` | Transfer learning | Reuse a pretrained network's generic filters instead of learning from nothing | LIACi images + pretrained ResNet18 |
| `NB16` | Autoencoder | Compress through a bottleneck; reconstruction error flags anomalies, unsupervised | Ship fuel data |

Notice what's *not* in this table: a single "best" architecture. `Each one answered a specific kind of question, on a specific kind of data`. That's the actual lesson of this block, made explicit today.

---

## 3. A decision framework for choosing an architecture

Three questions, asked in order, cover nearly everything from this course so far:

1. **Do you have labels?** If not, you're in `NB09`/`NB16` territory (clustering, PCA, autoencoders) — no architecture choice matters until this is answered.
2. **What shape is the data?** Tabular/structured (`NB07`/`NB08`/`NB11`), spatial/image (`NB13`/`NB15`), or sequential/time-dependent (`NB14`)? This determines the *family* of architecture, largely independent of how much data you have.
3. **How much data do you have?** This determines whether you train from scratch, reach for transfer learning, or skip Deep Learning entirely in favor of a classical model from `NB07`/`NB08`/`NB10` — the exact question this class experimentally tests in Part 5.

Let's draw that as a flowchart — deliberately shaped like the decision trees from `NB08`, since that's structurally exactly what this is:

Draw the decision framework as a flowchart:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(12, 7))

def box(x, y, w, h, text, color="lightblue"):
    ax.add_patch(patches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05",
                                         facecolor=color, edgecolor="black"))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=8.5)

def arrow(x1, y1, x2, y2, label=""):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", lw=1.3))
    if label:
        ax.text((x1 + x2) / 2 + 0.15, (y1 + y2) / 2, label, fontsize=8, color="darkblue")

box(4, 9, 3, 1, "Do you have\nlabels?", "khaki")
arrow(4, 9.5, 1, 7.5, "No")
arrow(6.5, 9.5, 8.5, 7.5, "Yes")

box(-0.5, 6.5, 3, 1, "Clustering / PCA (NB09)\nor Autoencoder (NB16)", "lightgreen")

box(7, 6.5, 3, 1, "What shape\nis the data?", "khaki")
arrow(7.2, 6.5, 4.5, 4.5, "Tabular")
arrow(8.5, 6.5, 8.5, 4.5, "Image")
arrow(9.8, 6.5, 12.5, 4.5, "Sequence")

box(2.5, 3.5, 3, 1, "How much\ndata?", "khaki")
arrow(2.8, 3.5, 0.5, 1.5, "Small/medium")
arrow(4.3, 3.5, 5.5, 1.5, "Large")

box(-0.7, 1, 3, 1, "Classical ML\n(NB07/NB08/NB10)", "lightcoral")
box(4.3, 1, 3, 1, "MLP, but compare\nto classical ML (NB11)", "lightcoral")

box(7, 3.5, 3, 1, "Enough data to\ntrain from scratch?", "khaki")
arrow(7.3, 3.5, 6, 1.5, "No")
arrow(8.7, 3.5, 10, 1.5, "Yes")
box(4.7, 1, 3, 1.3, "Transfer learning\n(NB15)", "lightcoral")
box(8.7, 1, 3, 1.3, "CNN from scratch\n(NB13)", "lightcoral")

box(11, 3.5, 3, 1.3, "RNN / LSTM\n(NB14)", "lightcoral")

ax.set_xlim(-1.5, 15)
ax.set_ylim(0, 10.5)
ax.axis("off")
ax.set_title("A decision framework for choosing an architecture")
plt.tight_layout()
plt.show()

This is a simplified guide, not a rigid rulebook — real problems sometimes sit between branches (tabular data with millions of rows might genuinely support an MLP; a small image dataset might do best with heavy data augmentation instead of transfer learning). `Treat it as a strong prior to start reasoning from, not a lookup table`.

> **Further reading**: [No free lunch theorem (Wikipedia)](https://en.wikipedia.org/wiki/No_free_lunch_theorem) — the formal reason no single algorithm dominates across every possible problem, and why a framework like this one has to stay a heuristic, not a law.

**Try it yourself**: turn the flowchart above into a real, callable function — plug in a hypothetical new problem's shape and see what it recommends.

In [ ]:
def suggest_architecture(has_labels, data_shape, n_examples):
    if not has_labels:
        return "Clustering/PCA (NB09) or Autoencoder (NB16)"
    if data_shape == "sequence":
        return "RNN/LSTM (NB14)"
    if data_shape == "image":
        return "CNN from scratch (NB13)" if n_examples >= 5000 else "Transfer learning (NB15)"
    # tabular
    return "Classical ML (NB07/NB08/NB10)" if n_examples < 5000 else "MLP, but compare to classical ML (NB11)"

print(suggest_architecture(has_labels=True, data_shape="image", n_examples=50000))
print(suggest_architecture(has_labels=True, data_shape="tabular", n_examples=100))


---

## 4. Applying the framework to every dataset used this block

Walk each real dataset from `NB11`–`NB16` through the three questions:

| Dataset | Labels? | Shape | Data volume | Framework says | What we actually did |
|---|---|---|---|---|---|
| Sonar (mine/rock) | Yes | Tabular (60 features) | 208 rows — small | Classical ML, or a modest MLP | `NB08`: classical ensembles/SVM. `NB11`/`NB12`: MLP, competitive but not clearly better |
| LIACi images | Yes | Image | 1,893 images — modest for images | Transfer learning likely beats from-scratch | `NB13`: from scratch. `NB15`: transfer learning — Part 5's logic predicts this should generally help |
| Ship fuel sequences | Yes | Sequential | 120 sequences — small | RNN/LSTM architecturally right, but data-hungry | `NB14`: LSTM vs. a naive baseline — a fair fight given how little data there was |
| Ship fuel data (anomalies) | No | Tabular | 1,440 rows | Unsupervised: clustering/PCA or autoencoder | `NB09`: K-Means/PCA. `NB16`: autoencoder |

The pattern worth noticing: **the framework's prediction and what each class's own honest evaluation actually found line up** — small tabular data made classical ML competitive in `NB08`/`NB11`, transfer learning had a real theoretical edge over the from-scratch CNN in `NB13`/`NB15`, and the LSTM in `NB14` never claimed an easy win because there wasn't enough data for one. `This isn't a coincidence — it's the framework doing its job`.

**Try it yourself**: run the actual datasets from this block's table through `suggest_architecture` — does the function's output match what the "Framework says" column above already claims?

In [ ]:
datasets = [
    ("Sonar (mine/rock)", True, "tabular", 208),
    ("LIACi images", True, "image", 1893),
    ("Ship fuel sequences", True, "sequence", 120),
    ("Ship fuel data (anomalies)", False, "tabular", 1440),
]

for name, has_labels, shape, n in datasets:
    print(f"{name}: {suggest_architecture(has_labels, shape, n)}")


---

## 5. Hands-on: does a neural network beat `NB10`'s tuned Gradient Boosting?

Time to stop asserting the "small tabular data favors classical ML" claim and actually test it. `NB10` built a complete, tuned project on the real **Yacht Hydrodynamics** dataset (308 real hull measurements) and reported a `GridSearchCV`-tuned Gradient Boosting model's test performance. Let's train a small MLP on the **exact same data and split**, and compare honestly.

In [ ]:
!wget -q -O yacht.data https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/yacht_hydrodynamics.data

import pandas as pd

columns = [
    "LongPos_COB", "Prismatic_Coeff", "LengthDisp_Ratio",
    "BeamDraft_Ratio", "LengthBeam_Ratio", "Froude_Number", "Residuary_Resistance",
]
yacht = pd.read_csv("yacht.data", sep=r"\s+", names=columns)

X = yacht.drop(columns="Residuary_Resistance")
y = yacht["Residuary_Resistance"]
print(yacht.shape)

Same split as `NB10` (`test_size=0.2, random_state=42`), so the comparison is apples to apples — no advantage from a lucky split on either side:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

A small MLP — deliberately similar in spirit to `NB11`'s `SonarMLP`, with dropout from `NB12`'s playbook to keep it honest on this little data:

In [ ]:
import torch.nn as nn

class YachtMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.layers(x)

torch.manual_seed(42)
yacht_mlp = YachtMLP(n_features=X_train_t.shape[1])

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(yacht_mlp.parameters(), lr=0.01)

n_epochs = 300
for epoch in range(n_epochs):
    yacht_mlp.train()
    optimizer.zero_grad()
    loss = criterion(yacht_mlp(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

print(f"Final training loss: {loss.item():.3f}")

Evaluate with `NB10`'s exact metrics:

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

yacht_mlp.eval()
with torch.no_grad():
    mlp_pred = yacht_mlp(X_test_t).numpy().ravel()

mlp_mae = mean_absolute_error(y_test, mlp_pred)
mlp_rmse = mean_squared_error(y_test, mlp_pred) ** 0.5
mlp_r2 = r2_score(y_test, mlp_pred)

print(f"MLP  - MAE: {mlp_mae:.3f}  RMSE: {mlp_rmse:.3f}  R2: {mlp_r2:.3f}")
print("Compare against NB10's tuned Gradient Boosting test-set numbers (re-run NB10 Part 8 if you don't have them handy).")

**Try it yourself**: don't just re-run `NB10` separately — replicate its exact tuned Gradient Boosting model here, on this same split, so the comparison is fully self-contained in one notebook.

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor

cv = KFold(n_splits=5, shuffle=True, random_state=42)
tuning_pipe = Pipeline([("scaler", StandardScaler()), ("model", GradientBoostingRegressor(random_state=42))])
param_grid = {
    "model__n_estimators": [100, 300, 500],
    "model__max_depth": [3, 5, 10],   # 3 is GradientBoostingRegressor's own default
    "model__min_samples_leaf": [1, 2, 4],
}
grid = GridSearchCV(tuning_pipe, param_grid, cv=cv, scoring="r2", n_jobs=-1)
grid.fit(X_train, y_train)

gb_pred = grid.best_estimator_.predict(X_test)
gb_mae = mean_absolute_error(y_test, gb_pred)
gb_rmse = mean_squared_error(y_test, gb_pred) ** 0.5
gb_r2 = r2_score(y_test, gb_pred)

print(f"Gradient Boosting (tuned here) - MAE: {gb_mae:.3f}  RMSE: {gb_rmse:.3f}  R2: {gb_r2:.3f}")
print(f"MLP (above)                     - MAE: {mlp_mae:.3f}  RMSE: {mlp_rmse:.3f}  R2: {mlp_r2:.3f}")


A bar chart makes the three-metric comparison easier to read at a glance:

In [ ]:
comparison = pd.DataFrame([
    {"Model": "MLP", "MAE": mlp_mae, "RMSE": mlp_rmse, "R2": mlp_r2},
    {"Model": "Gradient Boosting (tuned)", "MAE": gb_mae, "RMSE": gb_rmse, "R2": gb_r2},
])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, metric in zip(axes, ["MAE", "RMSE", "R2"]):
    ax.bar(comparison["Model"], comparison[metric], color=["steelblue", "darkorange"])
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


---

## 6. Interpreting the result

**Read your own numbers, honestly** — this section has three possible honest outcomes, and only one of them is the "expected" one:

- **Gradient Boosting wins**: the expected result, and good evidence for Part 3's framework — 308 rows is genuinely little data for a network with hundreds of trainable weights to learn from, while `NB10`'s `GridSearchCV` search could exploit the tree ensemble's much lower data-hunger.
- **They're close**: also a legitimate, common outcome — 6 clean, well-behaved numeric features with a fairly smooth relationship to the target is a relatively *easy* regression problem for both approaches.
- **The MLP wins**: less expected, but not impossible — worth a second look at whether the classical model was under-tuned, or whether this particular random seed's split happened to favor the network.

Whichever happened, you have **real evidence**, not a repeated claim — which is the entire point of running this comparison instead of just asserting it in a bullet point.

---

## 7. When Deep Learning *does* win: revisiting the evidence

Part 5 tested one side of the framework. The other side has evidence in this block too:

- `NB13`/`NB15`: the from-scratch CNN had to learn every filter — edges included — from ~400 images. Transfer learning started from filters already trained on 1.4 million images, needing far fewer training epochs to reach a comparable or better result on the same real task.
- `NB01` §3's original motivating table: Deep Learning's clearest wins are still on **raw, unstructured data** (images, audio, long sequences, text) — precisely the kind of data classical ML from Block 2 was never built to handle directly.

The honest, complete picture from this entire block: Deep Learning isn't "better" than classical ML in general — `it's a different set of tools, with a real advantage concentrated in specific situations` (unstructured data, large datasets, transfer learning availability) that Part 3's framework tries to name explicitly, rather than leaving as vague intuition.

---

## 8. Looking ahead: Block 4

**Block 4 — Proyectos: casos de estudio** applies everything from Blocks 2 and 3 to real, complete case studies using naval/ocean datasets not yet featured as a core teaching example in this course — weather data, historical voyage records, and terrain/elevation data. Each will mean choosing your own architecture using exactly the reasoning built today, not being told which one to use.

---

## 9. A checklist for approaching a new problem

Before writing a single line of model code on a new problem:

1. **Frame it.** What exactly are you predicting or discovering, and why does it matter? (`NB10` Part 2)
2. **Check for labels.** Supervised or unsupervised — this changes everything downstream. (Part 3, question 1)
3. **Know your data's shape.** Tabular, image, or sequential. (Part 3, question 2)
4. **Know your data's volume.** Rough order of magnitude is enough: tens, hundreds, thousands, millions. (Part 3, question 3)
5. **Start simple, then justify complexity.** A classical baseline (`NB07`/`NB08`) is fast to build and tells you whether a fancier model is even worth the effort — exactly what Part 5 just demonstrated directly.
6. **Evaluate honestly.** Held-out test data, touched once (`NB07` onward); compare against a naive baseline (`NB14`) or an existing result (today), never just your own training loss.
7. **Interpret, don't just report.** A confusion matrix, a feature-importance plot, a reconstruction-error crosstab — every real class in this course paired a number with an explanation of what it means operationally.

---

## Class summary

- Block 3 covered six real problems, each solved with the architecture that actually fit its data — not the same tool applied six times.
- Three questions (labels? data shape? data volume?) cover most of the architecture-choice decision, visualized as a flowchart shaped like `NB08`'s decision trees.
- We tested the "small tabular data favors classical ML" claim directly, training an MLP on `NB10`'s exact yacht dataset and split, rather than just asserting it.
- Deep Learning's real advantages concentrate on unstructured data, large datasets, and situations where transfer learning is available — not everywhere, all the time.
- Block 4 applies everything from Blocks 2–3 to new real case studies, with architecture choice left to you.

## Homework / Practice Ideas

1. Re-run Part 5 with `n_epochs` at 1000 instead of 300 — does the MLP close any gap with Gradient Boosting, or does it start overfitting on this small dataset instead (check by adding a validation split, `NB12`-style)?
2. Pick one dataset from Block 2 (`NB02`'s `Naval_Dataset.csv`, or `NB08`'s Sonar data) and walk it through Part 3's flowchart from scratch, writing down your reasoning at each branch before checking it against what the course actually did.
3. In your own words, explain why the Part 5 comparison used the *exact* same `random_state=42` split as `NB10` — what would be wrong with comparing against a different, new random split instead?
4. Using the checklist in Part 9, write out (in a markdown cell, no code required) how you would approach one of Block 4's upcoming topics (weather forecasting, historical voyage analysis, or terrain data) before that class ever gives you an answer.

> ***As always: a framework earns its keep by producing testable predictions — Part 5 is what makes this class's argument evidence, not opinion.***
